In [ ]:
import polars as pl

# 1. Загрузка исходных сырых данных (до генерации фичей)
data_path = "../data/raw/"
df_clients = pl.read_csv(f"{data_path}clients.csv", try_parse_dates=True)          # Справочник клиентов
df_monthly = pl.read_csv(f"{data_path}monthly_client.csv", try_parse_dates=True)   # Ежемесячная активность
df_nso     = pl.read_csv(f"{data_path}monthly_nso.csv", try_parse_dates=True)      # Данные по депозитам
df_kp = pl.read_csv(f"{data_path}kp.csv")

# 2. Агрегируем данные по депозитам (НСО) до уровня клиент + дата
nso_agg = df_nso.group_by(["client_id", "report_date"]).agg([
    pl.col("early_close_flag").max().alias("nso_early_close_any"),
    pl.col("outflow_amt").sum().alias("nso_outflow_sum"),
])

# 3. Объединяем ежемесячную активность со срочными вкладами и справочником
df_full_profile = (
    df_monthly
    .join(nso_agg, on=["client_id", "report_date"], how="left")
    .join(df_clients, on="client_id", how="left")
    .with_columns(
        pl.col("nso_early_close_any", "nso_outflow_sum").fill_null(0),
        # Расчет возраста клиента в месяцах на момент среза данных
        ((pl.col("report_date") - pl.col("registration_date")).dt.total_days() / 30.4).alias("client_age_months")
    )
)

# 4. Берем самый последний доступный месяц для каждого клиента, чтобы получить актуальный профиль
client_profiles_table = (
    df_full_profile
    .sort(["client_id", "report_date"])
    .unique(subset=["client_id"], keep="last") # Оставляет только последнюю хронологическую строчку для каждого клиента
    .drop(["registration_date", "cpr_id_right", "kp_id_right", "segment_name_right", "has_salary_project_right"]) # Удаляем техническую дату, так как посчитали возраст в месяцах
)
## Блок переноса названия отрасли в начало таблицы
client_profiles_table = client_profiles_table.join(df_kp, on="kp_id", how="left")
cols = client_profiles_table.columns
cols.remove("kp_name")                    # Убираем из конца
target_idx = cols.index("kp_id") + 1      # Находим индекс kp_id и смещаемся на 1 вправо
cols.insert(target_idx, "kp_name")        # Вставляем kp_name на нужное место
# 3. Применяем новый порядок
client_profiles_table = client_profiles_table.select(cols)
# Настройка отображения, чтобы Polars показал все столбцы профиля в Jupyter
with pl.Config(tbl_cols=-1, tbl_rows=10):
    print(f"Всего сформировано профилей для {client_profiles_table.height} уникальных клиентов.")
    display(client_profiles_table.head(20))
client_profiles_table.select(pl.col('client_id').n_unique())

Всего сформировано профилей для 3000 уникальных клиентов.


report_date,client_id,cpr_id,kp_id,kp_name,segment_name,demand_balance_start,demand_balance_end,demand_balance_avg,bal_min,bal_max,bal_median,bal_std,bal_cv,bal_days,nso_balance_start,nso_balance_end,nso_balance_avg,nso_interest_paid_amt,nso_principal_outflow_amt,nso_active_types_cnt,avg_rate,total_liabilities_end,txn_bal_amt,txn_cnt,txn_cnt_in,txn_cnt_out,txn_amt,txn_amt_in,txn_amt_out,txn_max_amt,txn_avg_amt,txn_median_amt,txn_std_amt,active_days,txn_self_amt,txn_other_amt,txn_int_amt,txn_ext_amt,txn_oth_bank_amt,uniq_ca_cnt,target_outflow_amt,target_outflow_cnt,op3_amt,op7_amt,op7_cnt,op10_amt,net_flow_amt,outflow_share,out_bal_amt,amt_pd,has_credit,active_credit_cnt,credit_limit_amt,credit_debt_amt,credit_payment_amt,credit_util,credit_overdue_flag,has_salary_project,salary_project_dt,salary_project_months,salary_employee_cnt,salary_fund_amt,salary_avg_payment_per_employee,salary_months_without_pay_1m,key_rate,month_num,month_sin,month_cos,is_quarter_end,nso_early_close_any,nso_outflow_sum,employee_cnt_start,client_age_months
date,str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,i64,f64,f64,f64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,i64,f64,i64,f64,f64,i64,f64,f64,f64,f64,f64,i64,i64,f64,f64,f64,f64,i64,i64,i64,i64,i64,f64,f64,i64,f64,i64,f64,f64,i64,i64,f64,i64,f64
2026-05-31,"""CL0000001""","""CPR000639""","""KP01""","""Добыча и сырье""","""MASS""",1.8106e6,2267723.7,2.0392e6,1.1530e6,2.3405e6,2267723.7,312726.67,0.1379,31,785127.69,794978.01,790052.85,13604.16,9421.22,1,20.33,3062701.7,2267723.7,35,16,19,3.5419e6,1.9995e6,1.5424e6,581382.51,101196.68,90769.12,69536.46,17,0.0,2.9225e6,1.0662e6,476134.34,238154.25,5,0.0,0,912704.4,0.0,0,0.0,457139.5,0.4355,1.5424e6,114254.31,0,0,0.0,0.0,0.0,0.0,0,0,0,0,0,0.0,0.0,0,16.967175,5,0.5,-0.866025,0,0,9421.22,0,150.098684
2026-05-31,"""CL0000002""","""CPR000581""","""KP10""","""АПК""","""Alfa Only""",3.9954e6,113730.1,2.0545e6,113730.1,5.2852e6,5.1991e6,1.9907e6,0.3829,31,3.1326e6,3.1416e6,3.1371e6,45520.95,36618.18,3,18.35,3.2553e6,113730.1,45,22,23,7.2639e6,1.6911e6,5572774.9,964015.32,161420.31,119229.83,152066.95,16,0.0,2.1007e6,5.2672e6,305584.12,208183.97,9,0.0,0,131383.12,0.0,0,5.0853e6,-3.8816e6,0.7672,5572774.9,234319.8,1,1,1.9313e6,404006.87,4170.05,0.2092,0,1,2005,66,115,5.0853e6,44220.29,0,16.967175,5,0.5,-0.866025,0,0,36618.18,129,315.493421
2026-05-31,"""CL0000003""","""CPR000075""","""KP08""","""Недвижимость""","""MASS""",4363898.6,136735.57,2.2503e6,0.0,5.3566e6,1.1034e6,1.9943e6,1.8075,31,0.0,0.0,0.0,0.0,0.0,0,null,136735.57,136735.57,56,27,29,9.1729e6,2.4729e6,6.7000e6,358594.86,163802.19,100773.54,184210.22,20,0.0,3.2835e6,6180050.7,519992.03,228811.05,7,0.0,0,474417.28,0.0,0,5.7096e6,-4.2272e6,0.7304,6.7000e6,295900.72,1,1,3.1013e6,642252.9,20131.07,0.2071,0,1,1112,37,97,5.7096e6,58861.36,0,16.967175,5,0.5,-0.866025,0,0,0.0,97,286.348684
2026-05-31,"""CL0000004""","""CPR000027""","""KP10""","""АПК""","""MASS""",2.3401e6,2.8268e6,2.5834e6,1.864312e6,2885049.3,2143341.1,379381.99,0.177,31,2.0151e6,2.0870e6,2.0510e6,32948.27,10727.86,3,19.37,4.9137e6,2.8268e6,38,15,23,3.1199e6,1.8033e6,1.3166e6,433878.92,82102.3,51578.0,98758.39,31,0.0,2.8186e6,496585.19,820018.65,273944.8,5,0.0,0,286086.52,0.0,0,0.0,486679.78,0.422,1.3166e6,100641.53,1,1,292301.63,147138.4,9575.87,0.5034,0,0,0,0,0,0.0,0.0,0,16.967175,5,0.5,-0.866025,0,0,10727.86,0,254.177632
2026-05-31,"""CL0000005""","""CPR000254""","""KP10""","""АПК""","""MASS""",578657.14,23021.21,300839.17,0.0,1.0270e6,578657.14,292711.42,0.5058,31,251072.65,253271.24,252171.94,3576.66,3452.96,1,17.68,276292.44,23021.21,34,14,20,1.7004e6,572403.15,1.1280e6,122566.68,50013.01,39124.03,54604.31,15,0.0,588225.0,1.1188e6,9250.36,7213.78,8,0.0,0,8156.36,0.0,0,1.0956e6,-555635.93,0.6634,1.1280e6,54852.98,1,3,787806.67,160204.6,931.0,0.2034,0,1,914,30,115,1.0956e6,9527.11,0,16.967175,5,0.5,-0.866025,0,0,3452.96,115,47.664474
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,

client_id
u32
3000


In [12]:
# 1. Проверяем точные названия столбцов и их типы в первой таблице
print("Таблица профилей:")
print(client_profiles_table.select(pl.col("kp_id")).schema)

# 2. Проверяем точные названия столбцов и их типы в файле kp.csv
print("\nСправочник kp.csv:")
print(df_kp.schema)

Таблица профилей:
Schema({'kp_id': String})

Справочник kp.csv:
Schema({'kp_id': String, 'kp_name': String})
